# MuseTalk 1.5 — T4 worker

One-click validation worker. Refuses CPU inference, downloads the official model files with `hf`, uses FP16 only for GPU inference models, and reports the real MuseTalk error instead of hiding it.

In [ ]:
# 1) Runtime + clean MuseTalk checkout
import os, sys, subprocess
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)
subprocess.run(['nvidia-smi'], check=False)
MT=Path('/content/MuseTalk'); VENV=Path('/content/musetalk310')
if not MT.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk.git',str(MT)],check=True)
if not (VENV/'bin/python').exists(): subprocess.run(['uv','venv','--python','3.10',str(VENV)],check=True)
PY=str(VENV/'bin/python')
subprocess.run([PY,'-m','pip','install','-q','--upgrade','pip','setuptools','wheel'],check=True)
subprocess.run([PY,'-c','import torch; print("torch=",torch.__version__); print("cuda_available=",torch.cuda.is_available()); print("cuda=",torch.version.cuda); print("gpu=",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'],check=True)
probe=subprocess.run([PY,'-c','import torch; print(torch.cuda.is_available())'],text=True,capture_output=True,check=True)
if probe.stdout.strip()!='True': raise RuntimeError('STOP: the MuseTalk Python environment cannot see CUDA/T4. Select Runtime > Change runtime type > T4 GPU, then run this notebook from the top.')
print('T4 CUDA runtime is ready — no CPU fallback will be allowed.')


In [ ]:
# 2) Exact MuseTalk dependencies + official weights
import subprocess, os
from pathlib import Path
PY='/content/musetalk310/bin/python'; HF='/content/musetalk310/bin/hf'; MT='/content/MuseTalk'; MODELS=Path(MT)/'models'; MODELS.mkdir(parents=True,exist_ok=True)
subprocess.run(['uv','pip','install','--python',PY,'pip','setuptools','wheel'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'torch==2.0.1','torchvision==0.15.2','torchaudio==2.0.2','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'-r',MT+'/requirements.txt'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'openmim','gdown'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'--no-build-isolation','chumpy==0.70'],check=True)
MIM='/content/musetalk310/bin/mim'
for pkg in ['mmengine','mmcv==2.0.1','mmdet==3.1.0','mmpose==1.1.0']: subprocess.run([MIM,'install',pkg],check=True)
def hf_download(repo, local_dir, *files):
    Path(local_dir).mkdir(parents=True,exist_ok=True)
    cmd=[HF,'download',repo,'--local-dir',str(local_dir)]
    for f in files: cmd += ['--include',f]
    subprocess.run(cmd,check=True)
hf_download('TMElyralab/MuseTalk', MODELS, 'musetalkV15/unet.pth','musetalkV15/musetalk.json')
hf_download('stabilityai/sd-vae-ft-mse', MODELS/'sd-vae', 'config.json','diffusion_pytorch_model.bin')
hf_download('openai/whisper-tiny', MODELS/'whisper', 'config.json','preprocessor_config.json','pytorch_model.bin')
hf_download('yzd-v/DWPose', MODELS/'dwpose', 'dw-ll_ucoco_384.pth')
hf_download('ManyOtherFunctions/face-parse-bisent', MODELS/'face-parse-bisent', '79999_iter.pth','resnet18-5c106cde.pth')
# Prevent Diffusers from trying the absent safetensors VAE file.
vae_py=Path(MT)/'musetalk/models/vae.py'; txt=vae_py.read_text(); txt=txt.replace('AutoencoderKL.from_pretrained(self.model_path)','AutoencoderKL.from_pretrained(self.model_path, use_safetensors=False)'); vae_py.write_text(txt)
print('Official MuseTalk model files are ready.')


In [ ]:
# 3) Upload approved singer image/video + ACE-Step audio, then run
from google.colab import files
from pathlib import Path
import subprocess, os, shutil
print('Upload the APPROVED singer image/video:')
avatar=next(iter(files.upload()))
print('Upload the successful ACE-Step bhajan MP3/WAV:')
audio=next(iter(files.upload()))
OUT=Path('/content/musetalk_output'); OUT.mkdir(parents=True,exist_ok=True)
avatar_src=OUT/'avatar_source.png'; audio_wav=OUT/'audio.wav'
subprocess.run(['ffmpeg','-y','-v','error','-i',avatar,'-frames:v','1','-vf','scale=512:-2','-pix_fmt','rgb24',str(avatar_src)],check=True)
subprocess.run(['ffmpeg','-y','-v','error','-i',audio,'-ar','16000','-ac','1',str(audio_wav)],check=True)
# Make upstream MuseTalk fail non-zero instead of swallowing processing exceptions.
inf_py=Path(MT)/'scripts/inference.py'
itxt=inf_py.read_text()
old='        except Exception as e:\n            print("Error occurred during processing:", e)\n'
new='        except Exception as e:\n            print("Error occurred during processing:", e)\n            raise\n'
if old not in itxt: raise RuntimeError('Unexpected MuseTalk inference.py layout; refusing to patch blindly.')
inf_py.write_text(itxt.replace(old,new,1))
print('MuseTalk exception handling patched: real failures will stop the cell.')
cfg=Path('/content/MuseTalk/configs/inference/test.yaml'); cfg.write_text(f'bhajan_test:
  video_path: "{avatar_src}"
  audio_path: "{audio_wav}"
  result_name: "bhajan_lipsync.mp4"
')
os.chdir('/content/MuseTalk')
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['PYTHONPATH']='/content/MuseTalk:'+env.get('PYTHONPATH',''); env['CUDA_VISIBLE_DEVICES']='0'
pre=subprocess.run(['/content/musetalk310/bin/python','-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'],text=True,capture_output=True)
if pre.returncode!=0: raise RuntimeError('STOP: MuseTalk Python cannot see the T4.\n'+pre.stderr)
print('Using GPU:',pre.stdout.strip())
result_dir=Path('/content/musetalk_output/result'); result_dir.mkdir(parents=True,exist_ok=True)
cmd=['/content/musetalk310/bin/python','-m','scripts.inference','--inference_config','configs/inference/test.yaml','--result_dir',str(result_dir),'--unet_model_path','models/musetalkV15/unet.pth','--unet_config','models/musetalkV15/musetalk.json','--whisper_dir','models/whisper','--version','v15','--fps','25','--batch_size','2','--use_float16','--parsing_mode','jaw']
print('Starting MuseTalk 1.5 on T4 — GPU FP16...')
r=subprocess.run(cmd,env=env,text=True,capture_output=True)
print(r.stdout)
if r.stderr: print(r.stderr)
candidates=list(result_dir.rglob('*.mp4'))
if r.returncode!=0 or not candidates: raise RuntimeError('MuseTalk did not create an MP4. The full STDERR/STDOUT is printed above; this is a real failure, not a silent loop.')
candidate=max(candidates,key=lambda p:p.stat().st_size)
if candidate.stat().st_size<100000: raise RuntimeError(f'Output MP4 is suspiciously small: {candidate}')
print('SUCCESS:',candidate); print('Size:',round(candidate.stat().st_size/1024/1024,2),'MB')
files.download(str(candidate))
